# 🤖 Entrenamiento y Comparación de Modelos de Detección de Anomalías

Este notebook entrena y compara **tres algoritmos** de detección de anomalías no supervisada
sobre el dataset `intel-cpu-dataset`, usando anomalías simuladas como verdad de referencia.

| Modelo | Enfoque |
|---|---|
| **Isolation Forest (IF)** | Aislamiento mediante árboles aleatorios |
| **Local Outlier Factor (LOF)** | Densidad local respecto a vecinos |
| **One-Class SVM** | Frontera de decisión con kernel RBF |

**Flujo:**
1. Instalación y carga
2. Preprocesamiento
3. Simulación de anomalías
4. Entrenamiento de los 3 modelos
5. Comparación de métricas
6. Visualizaciones comparativas
7. Selección del modelo ganador
8. Exportación del modelo

---
**Universidad ECCI · Electiva II — DevOps · 2026**
**Autores:** Julian David Garzon Medina · Javier Stiven Amaya Devia


## 1. Instalación y Carga

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib
print("✅ Dependencias listas")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, json, joblib, warnings
from datetime import datetime, timezone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

BLUE_DARK  = '#1F3864'
BLUE_MED   = '#2E75B6'
BLUE_LIGHT = '#9DC3E6'
ACCENT     = '#E74C3C'
GREEN      = '#27AE60'
ORANGE     = '#E67E22'
PURPLE     = '#8E44AD'

print("✅ Librerías importadas")


> Sube el archivo `intel_dataset.csv` cuando aparezca el botón.

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ Dataset cargado: {df_raw.shape[0]:,} registros × {df_raw.shape[1]} columnas")
df_raw.head()


## 2. Preprocesamiento

Imputación con mediana (conserva 2.081 registros) + StandardScaler.
Los modelos se entrenan **solo sobre datos normales**.


In [ ]:
NUM_COLS = [
    'cpu_usage', 'memory_usage', 'network_traffic',
    'power_consumption', 'num_executed_instructions',
    'execution_time', 'energy_efficiency'
]
MODEL_FEATURES = [
    'cpu_usage', 'memory_usage', 'network_traffic',
    'power_consumption', 'execution_time', 'energy_efficiency'
]

imputer = SimpleImputer(strategy='median')
df = df_raw.copy()
df[NUM_COLS] = imputer.fit_transform(df[NUM_COLS])

X_train = df[MODEL_FEATURES].copy()
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Registros de entrenamiento: {len(X_train):,}")
print(f"Features:                   {MODEL_FEATURES}")
print(f"Shape escalado:             {X_train_scaled.shape}")


## 3. Simulación de Anomalías

Generamos **100 registros anómalos conocidos** (25 por tipo) para usar como
verdad de referencia. Los modelos **no los ven durante el entrenamiento**.

| Tipo | CPU | Memory | Network | Representa |
|---|---|---|---|---|
| CPU/Mem Saturación | ≥92% | ≥90% | Normal | Proceso runaway, fuga de memoria |
| Network Spike | ≤8% | ≤12% | ≥950 | Exfiltración de datos, DDoS |
| Colapso Total | ≥97% | ≥97% | ≥950 | Fallo catastrófico |
| Fantasma | ≤0.5% | ≤0.5% | ≤2 | Servidor zombie, fallo silencioso |


In [ ]:
rng = np.random.default_rng(42)
N   = 25

anom_cpu_mem = pd.DataFrame({
    'cpu_usage':         rng.uniform(92, 100, N),
    'memory_usage':      rng.uniform(90, 100, N),
    'network_traffic':   rng.uniform(400, 650, N),
    'power_consumption': rng.uniform(380, 499, N),
    'execution_time':    rng.uniform(70,   99, N),
    'energy_efficiency': rng.uniform(0.8,   1, N),
})
anom_net_spike = pd.DataFrame({
    'cpu_usage':         rng.uniform(1,   8, N),
    'memory_usage':      rng.uniform(1,  12, N),
    'network_traffic':   rng.uniform(950,999, N),
    'power_consumption': rng.uniform(10,  40, N),
    'execution_time':    rng.uniform(0.5,  5, N),
    'energy_efficiency': rng.uniform(0.05,0.2,N),
})
anom_collapse = pd.DataFrame({
    'cpu_usage':         rng.uniform(97, 100, N),
    'memory_usage':      rng.uniform(97, 100, N),
    'network_traffic':   rng.uniform(950, 999, N),
    'power_consumption': rng.uniform(470, 499, N),
    'execution_time':    rng.uniform(92,   99, N),
    'energy_efficiency': rng.uniform(0.9,   1, N),
})
anom_ghost = pd.DataFrame({
    'cpu_usage':         rng.uniform(0.01, 0.5,   N),
    'memory_usage':      rng.uniform(0.01, 0.5,   N),
    'network_traffic':   rng.uniform(0.1,  2,     N),
    'power_consumption': rng.uniform(0.1,  1,     N),
    'execution_time':    rng.uniform(0.0001,0.01, N),
    'energy_efficiency': rng.uniform(0.0001,0.01, N),
})

tipos     = ['CPU/Mem Saturación', 'Network Spike', 'Colapso Total', 'Fantasma']
anomalies = pd.concat([anom_cpu_mem, anom_net_spike, anom_collapse, anom_ghost],
                       ignore_index=True)

X_full        = pd.concat([X_train, anomalies], ignore_index=True)
X_full_scaled = scaler.transform(X_full)
y_true        = np.array([1]*len(X_train) + [-1]*len(anomalies))

print(f"Registros normales:  {len(X_train):,}")
print(f"Anomalías simuladas: {len(anomalies)} (25 × 4 tipos)")
print(f"Dataset completo:    {len(X_full):,}")


## 4. Entrenamiento de los 3 Modelos

### 4.1 Isolation Forest

In [ ]:
t0 = time.time()
iso = IsolationForest(
    n_estimators=200, contamination=0.05,
    max_samples='auto', random_state=42, n_jobs=-1
)
iso.fit(X_train_scaled)
t_if_train = time.time() - t0

t0       = time.time()
pred_if  = iso.predict(X_full_scaled)
score_if = iso.decision_function(X_full_scaled)
t_if_inf = (time.time()-t0) / len(X_full) * 1000

print(f"✅ Isolation Forest entrenado en {t_if_train:.3f}s")
print(f"   Inferencia: {t_if_inf:.4f}ms por registro")


### 4.2 Local Outlier Factor (LOF)

In [ ]:
t0 = time.time()
lof = LocalOutlierFactor(
    n_neighbors=20, contamination=0.05,
    novelty=True, n_jobs=-1
)
lof.fit(X_train_scaled)
t_lof_train = time.time() - t0

t0        = time.time()
pred_lof  = lof.predict(X_full_scaled)
score_lof = lof.decision_function(X_full_scaled)
t_lof_inf = (time.time()-t0) / len(X_full) * 1000

print(f"✅ LOF entrenado en {t_lof_train:.3f}s")
print(f"   Inferencia: {t_lof_inf:.4f}ms por registro")


### 4.3 One-Class SVM

Aprende una **frontera de decisión** alrededor del comportamiento normal usando
un kernel RBF. Todo lo que queda fuera de la frontera se clasifica como anomalía.
El parámetro `nu` equivale a `contamination` en los otros modelos.


In [ ]:
t0 = time.time()
svm = OneClassSVM(
    kernel='rbf', nu=0.05, gamma='scale'
)
svm.fit(X_train_scaled)
t_svm_train = time.time() - t0

t0        = time.time()
pred_svm  = svm.predict(X_full_scaled)
score_svm = svm.decision_function(X_full_scaled)
t_svm_inf = (time.time()-t0) / len(X_full) * 1000

print(f"✅ One-Class SVM entrenado en {t_svm_train:.3f}s")
print(f"   Inferencia: {t_svm_inf:.4f}ms por registro")


## 5. Comparación de Métricas

In [ ]:
def calcular_metricas(pred, y_true, n_anomalias=100):
    mask_a = y_true == -1
    mask_n = y_true ==  1
    tp  = ((pred == -1) & mask_a).sum()
    fp  = ((pred == -1) & mask_n).sum()
    tn  = ((pred ==  1) & mask_n).sum()
    fn  = ((pred ==  1) & mask_a).sum()
    dr   = round(tp / n_anomalias * 100, 2)
    fpr  = round(fp / mask_n.sum() * 100, 2)
    prec = round(tp / (tp+fp) * 100, 2) if (tp+fp) > 0 else 0
    return {"TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn),
            "Tasa detección (%)":dr,
            "Falsos positivos (%)":fpr,
            "Precisión (%)":prec}

m_if  = calcular_metricas(pred_if,  y_true)
m_lof = calcular_metricas(pred_lof, y_true)
m_svm = calcular_metricas(pred_svm, y_true)

resumen = pd.DataFrame({
    'Métrica': [
        'Tasa de detección (%)', 'Falsos positivos (%)', 'Precisión (%)',
        'Tiempo entrenamiento (s)', 'Tiempo inferencia (ms/reg)',
        'TP', 'FP', 'FN'
    ],
    'Isolation Forest': [
        m_if['Tasa detección (%)'], m_if['Falsos positivos (%)'], m_if['Precisión (%)'],
        round(t_if_train,3), round(t_if_inf,4),
        m_if['TP'], m_if['FP'], m_if['FN']
    ],
    'LOF': [
        m_lof['Tasa detección (%)'], m_lof['Falsos positivos (%)'], m_lof['Precisión (%)'],
        round(t_lof_train,3), round(t_lof_inf,4),
        m_lof['TP'], m_lof['FP'], m_lof['FN']
    ],
    'One-Class SVM': [
        m_svm['Tasa detección (%)'], m_svm['Falsos positivos (%)'], m_svm['Precisión (%)'],
        round(t_svm_train,3), round(t_svm_inf,4),
        m_svm['TP'], m_svm['FP'], m_svm['FN']
    ],
})

print("COMPARACIÓN GENERAL:")
display(resumen)


In [ ]:
# Detección por tipo de anomalía
n_base = len(X_train)
resultados_tipo = []
for i, tipo in enumerate(tipos):
    s, e = n_base + i*25, n_base + (i+1)*25
    resultados_tipo.append({
        'Tipo':         tipo,
        'IF':           f"{(pred_if[s:e]==-1).sum()}/25  ({(pred_if[s:e]==-1).sum()/25*100:.0f}%)",
        'LOF':          f"{(pred_lof[s:e]==-1).sum()}/25  ({(pred_lof[s:e]==-1).sum()/25*100:.0f}%)",
        'One-Class SVM':f"{(pred_svm[s:e]==-1).sum()}/25  ({(pred_svm[s:e]==-1).sum()/25*100:.0f}%)",
    })

print("DETECCIÓN POR TIPO DE ANOMALÍA:")
display(pd.DataFrame(resultados_tipo))


## 6. Visualizaciones Comparativas

In [ ]:
# Gráfico 1: Métricas principales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación de Modelos — Métricas Principales', fontsize=13, fontweight='bold')

nombres   = ['IF', 'LOF', 'SVM']
colores   = [BLUE_MED, ORANGE, PURPLE]
titulos   = ['Tasa de Detección (%)
(↑ mejor)', 'Falsos Positivos (%)
(↓ mejor)', 'Precisión (%)
(↑ mejor)']
keys      = ['Tasa detección (%)', 'Falsos positivos (%)', 'Precisión (%)']
modelos_m = [m_if, m_lof, m_svm]

for ax, titulo, key in zip(axes, titulos, keys):
    vals = [m[key] for m in modelos_m]
    bars = ax.bar(nombres, vals, color=colores, alpha=0.85, edgecolor='white', width=0.5)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('%')
    ax.set_ylim(0, 115)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
                f'{val}%', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 2: Detección por tipo
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('Tasa de Detección por Tipo de Anomalía', fontsize=13, fontweight='bold')

x     = np.arange(len(tipos))
w     = 0.25
n_base = len(X_train)

pcts = {
    'IF':  [(pred_if[n_base+i*25:n_base+(i+1)*25]==-1).sum()/25*100  for i in range(4)],
    'LOF': [(pred_lof[n_base+i*25:n_base+(i+1)*25]==-1).sum()/25*100 for i in range(4)],
    'SVM': [(pred_svm[n_base+i*25:n_base+(i+1)*25]==-1).sum()/25*100 for i in range(4)],
}

for j, (nombre, color, offset) in enumerate(zip(nombres, colores, [-w, 0, w])):
    bars = ax.bar(x + offset, pcts[nombre], w, label=nombre,
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, pcts[nombre]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1,
                f'{val:.0f}%', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(tipos, fontsize=10)
ax.set_ylabel('% Detectadas')
ax.set_ylim(0, 118)
ax.axhline(80, color=ACCENT, linestyle='--', linewidth=1.5, label='Umbral mínimo 80%')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 3: Distribución del anomaly score
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribución del Anomaly Score — Normal vs Anomalía', fontsize=13, fontweight='bold')

mask_n = y_true ==  1
mask_a = y_true == -1

for ax, scores, titulo, color in zip(
    axes,
    [score_if, score_lof, score_svm],
    ['Isolation Forest', 'LOF', 'One-Class SVM'],
    [BLUE_MED, ORANGE, PURPLE]
):
    ax.hist(scores[mask_n], bins=50, alpha=0.6, color=color,
            label='Normal', density=True)
    ax.hist(scores[mask_a], bins=30, alpha=0.8, color=ACCENT,
            label='Anomalía simulada', density=True)
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Densidad')
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print("💡 Mayor separación entre distribuciones → mejor capacidad discriminativa.")


In [ ]:
# Gráfico 4: Scatter CPU vs Memory para los 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('CPU vs Memory — Anomalías detectadas por modelo', fontsize=13, fontweight='bold')

X_full_df = X_full.copy()

for ax, pred, titulo in zip(axes,
                             [pred_if, pred_lof, pred_svm],
                             ['Isolation Forest', 'LOF', 'One-Class SVM']):
    mask_tn = (pred ==  1) & (y_true ==  1)
    mask_tp = (pred == -1) & (y_true == -1)
    mask_fp = (pred == -1) & (y_true ==  1)
    mask_fn = (pred ==  1) & (y_true == -1)

    ax.scatter(X_full_df['cpu_usage'][mask_tn], X_full_df['memory_usage'][mask_tn],
               c=BLUE_LIGHT, s=8, alpha=0.3, label='Normal (TN)')
    ax.scatter(X_full_df['cpu_usage'][mask_tp], X_full_df['memory_usage'][mask_tp],
               c=ACCENT, s=30, alpha=0.8, zorder=5,
               label=f'Detectada TP={mask_tp.sum()}')
    ax.scatter(X_full_df['cpu_usage'][mask_fp], X_full_df['memory_usage'][mask_fp],
               c=ORANGE, s=20, alpha=0.7, zorder=4,
               label=f'Falso+ FP={mask_fp.sum()}')
    ax.scatter(X_full_df['cpu_usage'][mask_fn], X_full_df['memory_usage'][mask_fn],
               c=PURPLE, s=30, alpha=0.8, zorder=5, marker='x',
               label=f'No detectada FN={mask_fn.sum()}')
    ax.set_xlabel('CPU Usage (%)')
    ax.set_ylabel('Memory Usage (%)')
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=7, framealpha=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 5: Comparación multidimensional
categorias = ['Detección
(%)', 'Bajo FP
(100-FP%)',
              'Precisión
(%)', 'Vel.Train
(inv)', 'Vel.Inf
(inv)']

max_train = max(t_if_train, t_lof_train, t_svm_train)
max_inf   = max(t_if_inf,   t_lof_inf,   t_svm_inf)

def scores_modelo(m, t_train, t_inf):
    return [
        m['Tasa detección (%)'],
        round(100 - m['Falsos positivos (%)'], 2),
        m['Precisión (%)'],
        round((1 - t_train/max_train)*100, 1),
        round((1 - t_inf/max_inf)*100, 1),
    ]

all_scores = {
    'IF':  scores_modelo(m_if,  t_if_train,  t_if_inf),
    'LOF': scores_modelo(m_lof, t_lof_train, t_lof_inf),
    'SVM': scores_modelo(m_svm, t_svm_train, t_svm_inf),
}

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(categorias))
w = 0.25

for j, (nombre, color, offset) in enumerate(zip(nombres, colores, [-w, 0, w])):
    bars = ax.bar(x + offset, all_scores[nombre], w, label=nombre,
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, all_scores[nombre]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.8,
                f'{val:.0f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(categorias, fontsize=10)
ax.set_ylabel('Score (100 = mejor)')
ax.set_ylim(0, 118)
ax.set_title('Comparación Multidimensional (mayor = mejor)', fontweight='bold')
ax.axhline(80, color=ACCENT, linestyle='--', linewidth=1, alpha=0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


## 7. Selección del Modelo Ganador

In [ ]:
decision = pd.DataFrame({
    'Criterio': [
        'Tasa de detección', 'Falsos positivos', 'Precisión',
        'Velocidad entrenamiento', 'Velocidad inferencia',
        'CPU/Mem Saturación', 'Network Spike', 'Colapso Total', 'Fantasma',
        'Escalabilidad producción', 'Referencia estado del arte'
    ],
    'Isolation Forest': [
        f"{m_if['Tasa detección (%)']}%", f"{m_if['Falsos positivos (%)']}%",
        f"{m_if['Precisión (%)']}%", f"{t_if_train:.3f}s", f"{t_if_inf:.4f}ms",
        f"{(pred_if[n_base:n_base+25]==-1).sum()}/25",
        f"{(pred_if[n_base+25:n_base+50]==-1).sum()}/25",
        f"{(pred_if[n_base+50:n_base+75]==-1).sum()}/25",
        f"{(pred_if[n_base+75:n_base+100]==-1).sum()}/25",
        '✅ Alta — O(n log n)',
        'Carreño (2017), Chua (2024)'
    ],
    'LOF': [
        f"{m_lof['Tasa detección (%)']}%", f"{m_lof['Falsos positivos (%)']}%",
        f"{m_lof['Precisión (%)']}%", f"{t_lof_train:.3f}s", f"{t_lof_inf:.4f}ms",
        f"{(pred_lof[n_base:n_base+25]==-1).sum()}/25",
        f"{(pred_lof[n_base+25:n_base+50]==-1).sum()}/25",
        f"{(pred_lof[n_base+50:n_base+75]==-1).sum()}/25",
        f"{(pred_lof[n_base+75:n_base+100]==-1).sum()}/25",
        '⚠️ Media — O(n²)',
        'Ivars Carbó (2023), Chua (2024)'
    ],
    'One-Class SVM': [
        f"{m_svm['Tasa detección (%)']}%", f"{m_svm['Falsos positivos (%)']}%",
        f"{m_svm['Precisión (%)']}%", f"{t_svm_train:.3f}s", f"{t_svm_inf:.4f}ms",
        f"{(pred_svm[n_base:n_base+25]==-1).sum()}/25",
        f"{(pred_svm[n_base+25:n_base+50]==-1).sum()}/25",
        f"{(pred_svm[n_base+50:n_base+75]==-1).sum()}/25",
        f"{(pred_svm[n_base+75:n_base+100]==-1).sum()}/25",
        '⚠️ Baja — O(n²)–O(n³)',
        'Carreño (2017), Ivars Carbó (2023)'
    ],
})

display(decision)

print()
print("=" * 60)
print("VEREDICTO")
print("=" * 60)
print(f"  1° LOF           — Tasa detección: {m_lof['Tasa detección (%)']}%")
print(f"  2° One-Class SVM — Tasa detección: {m_svm['Tasa detección (%)']}%")
print(f"  3° Isolation Forest — Tasa detección: {m_if['Tasa detección (%)']}%")
print()
print("  Modelo seleccionado: LOF")
print("  Razón: mayor tasa de detección global (99%) con el menor")
print("  número de falsos positivos (4.1%), superando especialmente")
print("  en la detección de saturación CPU/Memoria (96% vs 76% SVM vs 68% IF).")
print()
print("  Nota: IF y SVM son superiores en escalabilidad para producción.")
print("  El reentrenamiento con datos reales puede cambiar el ranking.")

ganador = 'LOF'
modelo_ganador = lof


## 8. Exportación del Modelo

In [ ]:
version        = datetime.now(timezone.utc).strftime("v%Y%m%d_%H%M%S")
nombre_archivo = f"modelo_lof_{version}.pkl"

artefacto = {
    "model":    modelo_ganador,
    "scaler":   scaler,
    "features": MODEL_FEATURES,
    "version":  version,
    "algorithm":"LOF",
}

joblib.dump(artefacto, nombre_archivo)
joblib.dump(artefacto, "isolation_forest.pkl")  # nombre fijo para la API

metadata = {
    "version":        version,
    "trained_at":     datetime.now(timezone.utc).isoformat(),
    "model_selected": "LOF",
    "dataset": {
        "records_train":    len(X_train),
        "records_anomalies":len(anomalies),
        "features":         MODEL_FEATURES,
        "preprocessing":    "SimpleImputer(median) + StandardScaler",
    },
    "hyperparameters": {"n_neighbors": 20, "contamination": 0.05, "novelty": True},
    "metrics": {
        "tasa_deteccion_pct":   m_lof['Tasa detección (%)'],
        "falsos_positivos_pct": m_lof['Falsos positivos (%)'],
        "precision_pct":        m_lof['Precisión (%)'],
        "train_time_s":         round(t_lof_train, 4),
        "inf_ms_per_record":    round(t_lof_inf, 4),
    },
    "comparison": {
        "IF":  {"deteccion": m_if['Tasa detección (%)'],  "fp": m_if['Falsos positivos (%)']},
        "LOF": {"deteccion": m_lof['Tasa detección (%)'], "fp": m_lof['Falsos positivos (%)']},
        "SVM": {"deteccion": m_svm['Tasa detección (%)'], "fp": m_svm['Falsos positivos (%)']},
    },
    "detection_by_type": {
        t: int((pred_lof[n_base+i*25:n_base+(i+1)*25]==-1).sum())
        for i, t in enumerate(tipos)
    }
}

with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Modelo exportado: {nombre_archivo}")
print(f"✅ Copia fija:       isolation_forest.pkl")
print(f"✅ Metadatos:        metadata.json")
print()
print(json.dumps(metadata["metrics"], indent=2))


In [ ]:
# Descargar archivos
from google.colab import files

print("Descargando archivos — guárdalos en models/ del proyecto...")
files.download(nombre_archivo)
files.download("isolation_forest.pkl")
files.download("metadata.json")
print("✅ Listo — copia los 3 archivos a la carpeta models/ del proyecto")


## 9. Resumen Final

### Resultados obtenidos

| Modelo | Detección | Falsos + | Precisión | Inf (ms) |
|---|---|---|---|---|
| **LOF** ✅ | **99%** | **4.1%** | **53.5%** | 0.049ms |
| One-Class SVM | 94% | 4.9% | 48.0% | 0.007ms |
| Isolation Forest | 92% | 5.0% | 46.9% | 0.035ms |

### Por qué LOF gana con este dataset
LOF supera a IF y SVM porque el dataset tiene distribución quasi-uniforme
(kurtosis ≈ −1.2) y las anomalías son extremas y bien diferenciadas.
Con datos reales de producción (distribución orgánica con patrones),
IF puede recuperar terreno por su mayor robustez y escalabilidad.

### Próximos pasos
1. Copiar `isolation_forest.pkl` y `metadata.json` a `models/`
2. Levantar la API FastAPI con Docker
3. Conectar el nodo "Llamar API Modelo ML" en n8n
4. Publicar el workflow completo
